# 単語埋め込み（Word Embedding）

このノートブックでは、Transformer の最初のステップである
**単語埋め込み（Word Embedding）** を学びます。

テキスト（文字列）をコンピュータが計算できる数値（ベクトル）に変換する仕組みです。

## 目次
1. なぜ単語を数値に変換する必要があるのか？
2. トークン化（Tokenization）
3. 数理の基礎（ベクトル・転置・行列の積）
4. 単語埋め込みの仕組み（XW_e^T）
5. 具体例：7×6 行列を作る
6. なぜ多次元ベクトルが必要なのか？（多義語問題）
7. ベクトル空間での「意味の近さ」
8. まとめ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Transformer のどの部分を学んでいるか？（図4.9）
fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(-1, 15)
ax.set_ylim(-0.5, 2.5)
ax.axis('off')

blocks = [
    ('Input\nEmbedding', '#E65100', '#FFF3E0', True),
    ('Positional\nEncoding', '#9E9E9E', '#F5F5F5', False),
    ('Multi-Head\nAttention', '#9E9E9E', '#F5F5F5', False),
    ('Add &\nNorm', '#9E9E9E', '#F5F5F5', False),
    ('Feed\nForward', '#9E9E9E', '#F5F5F5', False),
    ('Add &\nNorm', '#9E9E9E', '#F5F5F5', False),
]

for i, (label, edge_color, face_color, highlight) in enumerate(blocks):
    x = i * 2.3
    lw = 3 if highlight else 1
    rect = mpatches.FancyBboxPatch((x, 0.3), 1.8, 1.5,
                                    boxstyle='round,pad=0.1',
                                    facecolor=face_color, edgecolor=edge_color, linewidth=lw)
    ax.add_patch(rect)
    ax.text(x + 0.9, 1.05, label, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(blocks) - 1:
        ax.annotate('', xy=(x + 2.1, 1.05), xytext=(x + 1.85, 1.05),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax.text(0.9, 2.2, '← 今ここ！', fontsize=12, fontweight='bold', color='#E65100', ha='center')
ax.set_title('エンコーダの構成 — Input Embedding を学ぶ', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1. なぜ単語を数値に変換する必要があるのか？

Transformer は数理モデルなので、内部はすべて **数値の計算** です。

しかし入力は `"Mount Fuji looks beautiful in spring."` のような **文字列** です。

文字列のままでは計算ができないので、最初に **単語を数値の列（ベクトル）に変換** する必要があります。

この変換は2段階で行われます：

```
文字列 → [トークン化] → トークンID列 → [単語埋め込み] → 数百次元のベクトル列
```

## 2. トークン化（Tokenization）

### 図4.10: 単語分割とトークン化

まず文章を**トークン**（最小単位）に分割し、各トークンに**ID（番号）**を割り当てます。

この機能を持つシステムを**トークナイザー（Tokenizer）**と呼びます。

| トークナイザーの特徴 | 説明 |
|---------------------|------|
| 分割方法が異なる | トークナイザーごとに単語の区切り方が違う |
| IDの振り方が異なる | 同じ単語でもトークナイザーによってIDが違う |
| あらかじめIDが決まっている | 各単語には事前にトークンIDが割り振られている |

### 図4.11: トークンIDから埋め込みベクトルへ

トークンIDが決まったら、**単語埋め込み（Word Embedding）**によって
各トークンを数百〜数万次元のベクトルに変換します。

In [ ]:
# 図4.10 & 4.11: トークン化 → 単語埋め込みの流れを可視化
# 書籍の例に基づくトークンID

tokens_with_ids = [
    ("Mount",     4057),
    ("Fuji",      20993),
    ("looks",     3504),
    ("beautiful", 3376),
    ("in",        1999),
    ("spring",    3500),
    (".",         1012),
]

# 書籍の埋め込みベクトルの例（先頭2次元のみ）
embedding_examples = [
    [0.3327, 0.2506],
    [0.4547, -0.3745],
    [0.1187, -0.4764],
    [0.4409, -0.2449],
    [-1.0407, -0.5206],
    [-0.3524, 0.3856],
    [-0.0428, -1.0624],
]

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(-0.5, len(tokens_with_ids) + 1)
ax.axis('off')
ax.set_title('図4.10 & 4.11: トークン化 → 単語埋め込み', fontsize=14, fontweight='bold')

# ヘッダー
ax.text(1.5, len(tokens_with_ids) + 0.3, '単語', ha='center', fontsize=11, fontweight='bold')
ax.text(4.5, len(tokens_with_ids) + 0.3, 'トークンID', ha='center', fontsize=11, fontweight='bold')
ax.text(10, len(tokens_with_ids) + 0.3, '埋め込みベクトル（数百〜数万次元）', ha='center', fontsize=11, fontweight='bold')

for i, ((word, tid), emb) in enumerate(zip(tokens_with_ids, embedding_examples)):
    y = len(tokens_with_ids) - 1 - i

    # 単語
    ax.text(1.5, y, word, ha='center', va='center', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#E3F2FD', edgecolor='gray'))

    # 矢印1
    ax.annotate('', xy=(3.2, y), xytext=(2.5, y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

    # トークンID
    ax.text(4.5, y, str(tid), ha='center', va='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFE0B2', edgecolor='#E65100'))

    # 矢印2
    ax.annotate('', xy=(6.0, y), xytext=(5.5, y),
                arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.5))

    # 埋め込みベクトル
    emb_text = f'[{emb[0]:7.4f}, {emb[1]:7.4f}, ...]'
    ax.text(10, y, emb_text, ha='center', va='center', fontsize=10, family='monospace',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#C8E6C9', edgecolor='#388E3C'))

# ラベル
ax.text(3.5, -0.7, 'Tokenization', ha='center', fontsize=10, style='italic', color='gray')
ax.text(7.5, -0.7, 'Word Embedding (W_e)', ha='center', fontsize=10, style='italic', color='gray')

plt.tight_layout()
plt.show()

print("ポイント:")
print("  1. 文章を単語（トークン）に分割する")
print("  2. 各トークンにID（番号）を割り当てる")
print("  3. 重み行列 W_e を使って各IDを数百次元のベクトルに変換する")
print("  4. W_e のパラメータは学習によって最適化される")

## 3. 数理の基礎（ベクトル・転置・行列の積）

単語埋め込みの数式を理解するために、必要な数学の道具を確認しましょう。

### 3.1 列ベクトルと転置ベクトル（図4.8）

7つのトークンを列ベクトル **X** で表すと：

$$\mathbf{X} = \begin{pmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \\ x_5 \\ x_6 \\ x_7 \end{pmatrix}, \quad \mathbf{X}^T = (x_1, x_2, x_3, x_4, x_5, x_6, x_7)$$

$T$ は **Transpose（転置）** の略で、縦を横に倒す操作です。

### 3.2 行列の積

$$AB = \begin{pmatrix} a_{1,1} & a_{1,2} \\ a_{2,1} & a_{2,2} \end{pmatrix} \begin{pmatrix} b_{1,1} & b_{1,2} \\ b_{2,1} & b_{2,2} \end{pmatrix} = \begin{pmatrix} a_{1,1}b_{1,1}+a_{1,2}b_{2,1} & a_{1,1}b_{1,2}+a_{1,2}b_{2,2} \\ a_{2,1}b_{1,1}+a_{2,2}b_{2,1} & a_{2,1}b_{1,2}+a_{2,2}b_{2,2} \end{pmatrix}$$

In [ ]:
# 列ベクトル、転置、行列の積を実際に計算

# === 書籍の例に合わせて7トークンの列ベクトル ===
print("=== 列ベクトルと転置 ===")
# x1〜x7 はトークンID（ここでは簡略化して1〜7）
X = np.array([1, 2, 3, 4, 5, 6, 7]).reshape(-1, 1)  # 7×1 列ベクトル
print(f"X（列ベクトル）: 形状 {X.shape}")
print(X)
print(f"\nX^T（転置）: 形状 {X.T.shape}")
print(X.T)

print()
print("=== 行列の積 ===")
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print(f"A =\n{A}")
print(f"\nB =\n{B}")

print(f"\nA × B の手計算:")
print(f"  左上: {A[0,0]}×{B[0,0]} + {A[0,1]}×{B[1,0]} = {A[0,0]*B[0,0] + A[0,1]*B[1,0]}")
print(f"  右上: {A[0,0]}×{B[0,1]} + {A[0,1]}×{B[1,1]} = {A[0,0]*B[0,1] + A[0,1]*B[1,1]}")
print(f"  左下: {A[1,0]}×{B[0,0]} + {A[1,1]}×{B[1,0]} = {A[1,0]*B[0,0] + A[1,1]*B[1,0]}")
print(f"  右下: {A[1,0]}×{B[0,1]} + {A[1,1]}×{B[1,1]} = {A[1,0]*B[0,1] + A[1,1]*B[1,1]}")
print(f"\nA × B =\n{A @ B}")

## 4. 単語埋め込みの仕組み（XW_e^T）

### 重みパラメータベクトル W_e

単語埋め込みには **重みパラメータベクトル W_e** が必要です。

$$\mathbf{W}_e = \begin{pmatrix} w_1 \\ w_2 \\ w_3 \\ w_4 \\ w_5 \\ w_6 \end{pmatrix}, \quad \mathbf{W}_e^T = (w_1, w_2, w_3, w_4, w_5, w_6)$$

- $\mathbf{W}_e$ の下付き $e$ は **embedding** の頭文字
- このパラメータは**学習によって最適化**される
- パラメータの値が出力精度を大きく左右する

### 行列積 XW_e^T

7トークンの列ベクトル（7×1）と転置した重みベクトル（1×6）の行列積を計算すると **7×6 の行列** ができます：

$$\mathbf{X}\mathbf{W}_e^T = \begin{pmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \\ x_5 \\ x_6 \\ x_7 \end{pmatrix} (w_1, w_2, w_3, w_4, w_5, w_6) = \begin{pmatrix} x_1 w_1 & x_1 w_2 & \cdots & x_1 w_6 \\ x_2 w_1 & x_2 w_2 & \cdots & x_2 w_6 \\ \vdots & \vdots & \ddots & \vdots \\ x_7 w_1 & x_7 w_2 & \cdots & x_7 w_6 \end{pmatrix}$$

この 7×6 行列の各行が、各トークンの**埋め込みベクトル**（6次元）です。

In [ ]:
# 書籍に合わせて 7トークン × 6次元 で計算

words_full = ["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]
n_tokens = 7   # トークン数
embed_dim = 6   # 埋め込み次元（書籍の設定）

print("=== XW_e^T の計算 ===")
print(f"トークン数: {n_tokens}（ピリオド含む）")
print(f"埋め込み次元: {embed_dim}（書籍の設定。実際は数百〜数万）")
print()

# トークンIDに対応する値（簡略化のため 0〜6 の整数）
X = np.arange(1, n_tokens + 1).reshape(-1, 1)  # 7×1
print(f"X（7×1 列ベクトル）:")
for i, word in enumerate(words_full):
    print(f"  x{i+1} = {X[i,0]}  ← {word}")

# 重みパラメータベクトル
np.random.seed(42)
W_e = np.round(np.random.randn(embed_dim, 1) * 0.5, 2)  # 6×1
print(f"\nW_e（6×1 列ベクトル）:")
for i in range(embed_dim):
    print(f"  w{i+1} = {W_e[i,0]:6.2f}")

print(f"\nW_e^T（1×6 転置ベクトル）:")
print(f"  {W_e.T[0]}")

# 行列積
embedding_matrix = X @ W_e.T  # 7×1 × 1×6 = 7×6
print(f"\nXW_e^T（7×6 行列）:")
print(f"形状: {embedding_matrix.shape}")
print()
header = f"{'':12s}" + "".join(f"{'w'+str(i+1):>8s}" for i in range(embed_dim))
print(header)
print("-" * len(header))
for i, word in enumerate(words_full):
    vals = "".join(f"{v:8.2f}" for v in embedding_matrix[i])
    print(f"{word:12s}{vals}")

## 5. 具体例：7×6 行列を作る（図4.13）

実際の Transformer では、各トークンIDごとに重み行列の対応する行を取り出します。

ここでは学習済みの重み行列を使って、より現実的な埋め込みを行いましょう。

In [ ]:
# 図4.13: 7×6 行列の可視化
# 実際の学習済みモデルに近い値を設定

np.random.seed(123)
# 語彙全体の埋め込み行列（実際は数万行×数百列）から
# 各トークンIDに対応する行を取り出す
embedding_realistic = np.round(np.random.randn(n_tokens, embed_dim) * 0.5, 4)

print("=== 図4.13: 7×6 埋め込み行列 ===")
print("各行 = 1つのトークンの埋め込みベクトル（6次元）")
print("各列 = 1つの次元")
print()

header = f"{'':12s}" + "".join(f"{'d'+str(i+1):>9s}" for i in range(embed_dim))
print(header)
print("-" * len(header))
for i, word in enumerate(words_full):
    vals = "".join(f"{v:9.4f}" for v in embedding_realistic[i])
    print(f"{word:12s}{vals}")

print(f"\n行列の形状: {embedding_realistic.shape}  （{n_tokens}トークン × {embed_dim}次元）")

In [ ]:
# 7×6 行列をヒートマップで可視化

fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(embedding_realistic, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

for i in range(n_tokens):
    for j in range(embed_dim):
        val = embedding_realistic[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=10, color=color)

ax.set_yticks(range(n_tokens))
ax.set_yticklabels(words_full, fontsize=12)
ax.set_xticks(range(embed_dim))
ax.set_xticklabels([f'次元{i+1}' for i in range(embed_dim)], fontsize=11)
ax.set_title('図4.13: 単語埋め込み行列（7×6）\nInput Embedding の出力', fontsize=14, fontweight='bold')
ax.set_ylabel('7 トークン', fontsize=12)
ax.set_xlabel('6 次元', fontsize=12)

plt.colorbar(im, ax=ax, label='値')
plt.tight_layout()
plt.show()

print("この 7×6 行列が Transformer のエンコーダの入力データとなる。")
print("次の処理は位置符号化（Positional Encoding）の加算。")

## 6. なぜ多次元ベクトルが必要なのか？（多義語問題）

単語をトークン化して数値IDにするだけではダメなのでしょうか？
なぜわざわざ数百〜数万次元のベクトルに変換するのでしょうか？

その理由の1つは **多義語への対応** です。

| トークン | 意味1 | 意味2 | 意味3 | 意味4 |
|---------|-------|-------|-------|-------|
| **looks** | 見る | 似ている | 〜のように見える | ー |
| **spring** | 春 | 泉 | ばね | 跳ねる |

たった1つの数値（ID）では、これらすべての意味を表現できません。

数百〜数万次元のベクトルにすることで、1つのトークンを**多角的に解釈**できるようになります。

In [ ]:
# 多義語の問題を可視化

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# === 左: 1次元（ID）では不十分 ===
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('1次元（ID）の場合\n多義語を区別できない', fontsize=12, fontweight='bold', color='red')

# "spring" のID
ax.text(5, 4, '"spring" → ID: 3500', ha='center', fontsize=14, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FFCDD2', edgecolor='red'))

meanings = ['春', '泉', 'ばね', '跳ねる', '湧き出る']
for i, meaning in enumerate(meanings):
    ax.text(2 + i * 1.5, 2, meaning, ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='#FFF3E0', edgecolor='gray'))
    ax.annotate('', xy=(2 + i * 1.5, 2.5), xytext=(5, 3.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax.text(5, 1, '→ 1つの数値で5つの意味を\n  区別するのは不可能！', ha='center', fontsize=11, color='red')

# === 右: 多次元ベクトルなら区別可能 ===
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('多次元ベクトルの場合\n多角的に意味を表現できる', fontsize=12, fontweight='bold', color='green')

ax.text(5, 4, '"spring" → [0.12, -0.45, 0.78, ...]', ha='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#C8E6C9', edgecolor='green'))

dims = ['季節の次元\n→ 「春」', '自然の次元\n→ 「泉」', '物理の次元\n→ 「ばね」', '動作の次元\n→ 「跳ねる」']
for i, dim in enumerate(dims):
    ax.text(1.5 + i * 2.2, 2, dim, ha='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#E8F5E9', edgecolor='gray'))
    ax.annotate('', xy=(1.5 + i * 2.2, 2.7), xytext=(5, 3.5),
                arrowprops=dict(arrowstyle='->', color='green', lw=1))

ax.text(5, 1, '→ 各次元が異なる意味の側面を\n  捉えることで多義語に対応！', ha='center', fontsize=11, color='green')

plt.tight_layout()
plt.show()

## 7. ベクトル空間での「意味の近さ」（図4.12）

単語を多次元ベクトルに変換すると、**意味が近い単語どうしのベクトルが近くなる**という性質が得られます。

書籍の図4.12 では、king, queen, man, woman の4つの単語を2次元で可視化しています：
- [king] と [queen] が近い（王族）
- [man] と [woman] が近い（一般人）

ベクトル同士の「近さ」を計算するには、第2章で学んだ **内積** や **コサイン類似度** を使います。
Transformer では **内積が極めて重要な役割** を果たします（後のノートブックで詳しく学びます）。

In [ ]:
# 図4.12: king, queen, man, woman のベクトル空間での配置
# 書籍の値に基づく

word_vectors = {
    'king':  np.array([2, 5]),
    'queen': np.array([3, 4]),
    'woman': np.array([4, 2]),
    'man':   np.array([3, 1]),
}

fig, ax = plt.subplots(figsize=(8, 7))

colors = {'king': '#1565C0', 'queen': '#C62828', 'woman': '#C62828', 'man': '#1565C0'}

# 各単語のベクトルを矢印で描画
for word, vec in word_vectors.items():
    ax.annotate('', xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors[word], lw=2))
    ax.text(vec[0] + 0.15, vec[1] + 0.15, f'{word}\n({vec[0]},{vec[1]})',
            fontsize=12, fontweight='bold', color=colors[word])

# 「近い」ことを示す破線
ax.plot([2, 3], [5, 4], '--', color='gray', alpha=0.5, lw=1.5)
ax.text(2.3, 4.7, '近い（王族）', fontsize=9, color='gray')

ax.plot([3, 4], [1, 2], '--', color='gray', alpha=0.5, lw=1.5)
ax.text(3.3, 1.3, '近い（一般人）', fontsize=9, color='gray')

ax.set_xlim(-0.5, 6)
ax.set_ylim(-0.5, 6)
ax.set_xlabel('Dimension 1', fontsize=12)
ax.set_ylabel('Dimension 2', fontsize=12)
ax.set_title('図4.12: 単語ベクトルの2次元可視化\n意味が近い単語は空間上でも近い', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

# コサイン類似度を計算
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("=== コサイン類似度 ===")
pairs = [('king', 'queen'), ('man', 'woman'), ('king', 'man'), ('queen', 'woman')]
for w1, w2 in pairs:
    sim = cosine_similarity(word_vectors[w1], word_vectors[w2])
    print(f"  {w1:6s} ↔ {w2:6s}: {sim:.4f}")

print()
print("→ king-queen、man-woman の類似度が高い")
print("→ 実際は数百次元だが、2次元でもイメージが掴める")
print("→ Transformer では内積（ベクトルの近さ）が重要な役割を果たす")

## 8. まとめ

| ポイント | 内容 |
|----------|------|
| **トークン化** | 文章を単語（トークン）に分割し、IDを割り当てる |
| **トークナイザー** | トークン化を行うシステム。種類によってID が異なる |
| **単語埋め込み** | トークンIDを数百〜数万次元のベクトルに変換する |
| **W_e（重みパラメータ）** | 学習によって最適化される。出力精度を大きく左右する |
| **XW_e^T** | 7×1 × 1×6 = 7×6 行列（トークン数×埋め込み次元） |
| **多義語対応** | 1つの数値では多義語の意味を区別できない → 多次元で対応 |
| **意味の近さ** | 学習後は意味が近い単語のベクトルが空間上でも近くなる |
| **内積** | ベクトル間の近さの指標。Transformer で極めて重要 |

### この段階での出力

```
入力文: "Mount Fuji looks beautiful in spring."
  ↓ トークン化（Tokenization）
["Mount", "Fuji", "looks", "beautiful", "in", "spring", "."]
  ↓ 単語埋め込み（XW_e^T）
7×6 の行列（7トークン × 6次元ベクトル）
```

## 次のステップ

次のノートブックでは **位置符号化（Positional Encoding）** を学びます。

単語埋め込みだけでは **単語の順番の情報** が失われてしまいます。
位置符号化の値を 7×6 行列に**加算**することで、順番の情報を付与します。